# 1 Libraries

In [ ]:
from mxnet import nd, gluon, init, autograd, gpu
from mxnet.gluon import nn
import mxnet as mx

import h5py
import numpy as np
import math
import time
import datetime
from os import makedirs
import os

# 2 Constants

In [ ]:
root = ''
file_name = 'Data_for_NSU_32v2.mat'

results_path = ''

gray_symbols_16qam = np.sqrt(0.1) * np.array(
                    [1+1j, 1+3j, 1-1j, 1-3j, 
                    3+1j, 3+3j, 3-1j, 3-3j, 
                    -1+1j, -1+3j, -1-1j, -1-3j,
                    -3+1j, -3+3j, -3-1j, -3-3j])

swap_64_to_128_complex = False

if swap_64_to_128_complex:
    complex_t = np.complex128
    real_t = np.float64
else:
    complex_t = np.complex64
    real_t = np.float32

# 3 Load .mat file

In [ ]:
with h5py.File(root + file_name, 'r') as f:
    BER_CDC_X = np.array(f['BER_CDC_X'])[0]
    BER_CDC_Y = np.array(f['BER_CDC_Y'])[0]
    BER_DBP_X = np.array(f['BER_DBP_X'])[0]
    BER_DBP_Y = np.array(f['BER_DBP_Y'])[0]

    RX_CDC = np.asarray((f['RX_CDC'][()])['real'], dtype=real_t) + 1j * np.asarray((f['RX_CDC'][()])['imag'], dtype=real_t)
    RY_CDC = np.asarray((f['RY_CDC'][()])['real'], dtype=real_t) + 1j * np.asarray((f['RY_CDC'][()])['imag'], dtype=real_t)
    TX = np.asarray((f['TX'][()])['real'], dtype=real_t) + 1j * np.asarray((f['TX'][()])['imag'], dtype=real_t)
    TY = np.asarray((f['TY'][()])['real'], dtype=real_t) + 1j * np.asarray((f['TY'][()])['imag'], dtype=real_t)

    Dcd = (f['param_ch']['Dcd'])[()].item()
    fiberL = (f['param_ch']['fiberL'])[()].item()
    gamma = (f['param_ch']['gamma'])[()].item()
    lmbda = (f['param_ch']['lambda'])[()].item()
    
    Fn = np.array(f['param_gen']['Fn']).T[0]
    Fsym = (f['param_gen']['Fsym'])[()].item()
    Po = (f['param_gen']['Po'])[()].item()


Dcd = Dcd * 1e-6
fiberL = fiberL * 1e3
lmbda = lmbda * 1e-9
gamma = gamma * 1e-3

c = 299792458
b2 = -lmbda ** 2 * Dcd / ( 2 * math.pi * c)
power = 10 ** ((Po-30)/10) / 2

# 4 Variables

# 4.1 Defined

In [ ]:
ctx = gpu(0)
#ctx = mx.cpu(0)

channel_numbers = [0, 1, 2, 3]

train_portion = 0.5
CDC_data_size = 2 ** 17

num_step = 19
filt_CDC_width = 101

manual_shift = True

filt_CDC_last_width = 71
filt_FD_width = 21

nl_mem = 6
nl_mem_nc1_1 = 7
nl_mem_nc1_2 = 15
nl_mem_nc2_1 = 1
nl_mem_nc2_2 = 1
nl_mem_nc3_1 = 1
nl_mem_nc3_2 = 1

epochs = 2000
batch_size = 900000
lrate = 1e-3
lrate_CDC = 1e-3
    
sym_filt = True
sym_nonlin = True

save_model_best = True
model_filename = "net.params"

# 4.2 Calculated

In [ ]:
channels = len(channel_numbers)
data_dimension = 2 * 2 * channels

central_frequency = np.sum(Fn[channel_numbers]) / channels
Fn = Fn - central_frequency

data_size = RX_CDC.shape[1]
train_data_size = int(data_size * train_portion)
CDC_data_size = min(CDC_data_size, train_data_size)

filt_CDC_delay = int((filt_CDC_width-1)/2)
filt_CDC_last_delay = int((filt_CDC_last_width-1)/2)
filt_FD_delay = int((filt_FD_width-1)/2)

if sym_filt:
    filt_CDC_width = int((filt_CDC_width + 1) / 2)
    filt_CDC_last_width = int((filt_CDC_last_width + 1) / 2)

symbol_shift = (fiberL / num_step * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
if np.max(symbol_shift) == 0 or not manual_shift:
    step_length = fiberL / num_step
    last_step_length = 0
    FD_step_length = 0
else:
    symbol_shift_distance = np.max(abs(1/(2 * math.pi * b2 * Fsym * Fn[channel_numbers])))
    ind = np.argmax(abs(1/(2 * math.pi * b2 * Fsym * Fn[channel_numbers])))
    step_length = symbol_shift[ind] * symbol_shift_distance
    symbol_shift = (step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
    last_symbol_shift = ((fiberL - step_length * num_step) * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
    last_step_length = last_symbol_shift[ind] * symbol_shift_distance
    last_symbol_shift = (last_step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
    FD_step_length = fiberL - step_length * num_step - last_step_length

step_shifts = np.zeros((channels,2),dtype=int)
if last_step_length > 0:
    last_shifts = np.zeros((channels,2),dtype=int)
    
for i in range(channels):
    if symbol_shift[i] > 0:
        step_shifts[i,0] = abs(np.min(symbol_shift)) - symbol_shift[i]
        step_shifts[i,1] = np.max(symbol_shift) + symbol_shift[i]
    else:
        step_shifts[i,0] = np.max(symbol_shift) - symbol_shift[i]
        step_shifts[i,1] = abs(np.min(symbol_shift)) + symbol_shift[i]
    
    if last_step_length > 0:
        if last_symbol_shift[i] > 0:
            last_shifts[i,0] = abs(np.min(last_symbol_shift)) - last_symbol_shift[i]
            last_shifts[i,1] = np.max(last_symbol_shift) + last_symbol_shift[i]
        else:
            last_shifts[i,0] = np.max(last_symbol_shift) - last_symbol_shift[i]
            last_shifts[i,1] = abs(np.min(last_symbol_shift)) + last_symbol_shift[i]

step_output_shifts = np.zeros((2,),dtype=int)
full_output_shifts = np.zeros((2,),dtype=int)

if last_step_length > 0:
    last_output_shifts = np.zeros((2,),dtype=int)

            
step_output_shifts[0] = abs(np.min(symbol_shift))
step_output_shifts[1] = np.max(symbol_shift)  
if last_step_length > 0:
    last_output_shifts[0] = abs(np.min(last_symbol_shift))
    last_output_shifts[1] = np.max(last_symbol_shift)  

full_output_shifts[0] = num_step * abs(np.min(symbol_shift))
full_output_shifts[1] = num_step * np.max(symbol_shift)
if last_step_length > 0:
    full_output_shifts[0] += abs(np.min(last_symbol_shift))
    full_output_shifts[1] += np.max(last_symbol_shift)

max_full_shift = int(np.sum(full_output_shifts))

nonlin_coef = gamma * power * 0.05 * 8/9 * step_length / channels

if sym_nonlin:
    memory_size = nl_mem + 1
else:
    memory_size = 2 * nl_mem + 1
memory_size_nc1 = nl_mem_nc1_1 + nl_mem_nc1_2 + 1
memory_size_nc2 = nl_mem_nc2_1 + nl_mem_nc2_2 + 1
memory_size_nc3 = nl_mem_nc3_1 + nl_mem_nc3_2 + 1

nl_mem_nc = np.zeros((4,2),dtype=int)
nl_mem_nc[0,0] = nl_mem
nl_mem_nc[0,1] = nl_mem
nl_mem_nc[1,0] = nl_mem_nc1_1
nl_mem_nc[1,1] = nl_mem_nc1_2
nl_mem_nc[2,0] = nl_mem_nc2_1
nl_mem_nc[2,1] = nl_mem_nc2_2
nl_mem_nc[3,0] = nl_mem_nc3_1
nl_mem_nc[3,1] = nl_mem_nc3_2

max_nl_shift = np.max(nl_mem_nc[:channels,:])

nl_shifts = np.zeros((channels,channels,2),dtype=int)
for i in range(channels):
    for j in range(channels):
        if i == j:
            nl_shifts[i,i,0] = max_nl_shift - nl_mem
            nl_shifts[i,i,1] = max_nl_shift - nl_mem
        elif i > j:
            nl_shifts[i,j,1] = max_nl_shift - nl_mem_nc[i-j,1]
            nl_shifts[i,j,0] = max_nl_shift - nl_mem_nc[i-j,0]
        else:
            nl_shifts[i,j,1] = max_nl_shift - nl_mem_nc[j-i,0]
            nl_shifts[i,j,0] = max_nl_shift - nl_mem_nc[j-i,1]
        
full_delay = num_step * (filt_CDC_delay + max_nl_shift)
if last_step_length > 0:
    full_delay += filt_CDC_last_delay
if FD_step_length > 0:
    full_delay += filt_FD_delay

# 5 Functions

# 5.1 BER and CD

In [ ]:
def symbols_to_codes(data_complex):
    codes = np.zeros(len(data_complex), dtype=np.int32)
    for i in range(len(data_complex)):
        codes[i] = np.argmin(abs(gray_symbols_16qam - data_complex[i]))
    
    return codes

def ber_by_codes(tx, rx):
    diff = tx ^ rx
    errors = 0
    for error in diff:
        while error:
            error &= error - 1
            errors +=1
            
    return 0.25 * errors / len(diff)

def calculate_ber(tx, rx):
    return ber_by_codes(symbols_to_codes(tx[0,0,:] + 1j * tx[0,1,:]), symbols_to_codes(rx[0,0,:] + 1j * rx[0,1,:]))

def demapper(tx, rx):
    phi = (1+np.sqrt(5))/2
    xl = 0.9
    xr = 1.1
    for j in range(10):
        x1 = xr - (xr-xl)/phi
        x2 = xl + (xr-xl)/phi
        y1 = calculate_ber(tx, rx * x1)
        y2 = calculate_ber(tx, rx * x2)
        if y1 >= y2:
            xl = x1
        else:
            xr = x2
        x = (x1+x2)/2
    
    return x

def cd_operator(data, size, length, chFreq, direction):
    dw = 2*math.pi*Fsym/size
    w = np.arange(-size/2, size/2,1) * dw
    w = np.fft.fftshift(w)
    
    if direction == 'f':
        s = 1
    if direction == 'b':
        s = -1
    
    fft_data = np.fft.fft(data)
    fft_data = fft_data * np.exp(s * 1j * b2 / 2 * (w + 2 * math.pi * chFreq) ** 2 * length)
    dataCD = np.fft.ifft(fft_data)

    return dataCD

def create_metadata():
    metadata1 = dict( (name, eval(name)) for name in ['channels', 'channel_numbers', 'train_portion', 'train_data_size', 
                                                      'CDC_data_size', 'num_step', 'filt_CDC_width', 'nl_mem', 'nl_mem_nc1_1', 
                                                      'nl_mem_nc1_2', 'nl_mem_nc2_1', 'nl_mem_nc2_2', 'nl_mem_nc3_1', 'nl_mem_nc3_2'])
    metadata2 = dict( (name, eval(name)) for name in ['epochs', 'batch_size','lrate', 'lrate_CDC', 'manual_shift', 'sym_filt', 
                                                      'sym_nonlin', 'save_model_best'])

    now = datetime.datetime.now()
    date_and_time = now.strftime('%Y%m%d_%H%M')
    dir_path = results_path + date_and_time + '/'
    makedirs(dir_path)
    
    f = open(dir_path + 'metadata.md', 'w')
    f.write(now.strftime('%Y-%m-%d %H:%M') + '\n\n')
    
    for x, y in metadata1.items():
        f.write('{}: {}\n'.format(x, y))
    if manual_shift:
        f.write('filt_CDC_last_width: {}\n'.format(filt_CDC_last_width))
        f.write('filt_FD_width: {}\n'.format(filt_FD_width))
    for x, y in metadata2.items():
        f.write('{}: {}\n'.format(x, y))
    f.close()
    
    return dir_path

# 5.2 Initialization and Dataloader

In [ ]:
class CustomInit(mx.init.Initializer):
    def __init__(self, filt):
        super(CustomInit, self).__init__()
        self.filt = filt
        
    def _init_weight(self, _, arr):
        arr[:] = self.filt

def dataloader(X, y, batch, enableShuffle):
    size = X.shape[1]
    batch = min(batch, size)
    d = size / batch
    if d%1 > 0.2:
        batch = np.floor(size / np.ceil(d)).astype(int)
    batch = batch - batch%2
    number_of_bathes = np.floor(size / batch).astype(int)
    
    data = nd.empty((number_of_bathes, data_dimension, batch))
    if manual_shift:
        label = nd.empty((number_of_bathes, data_dimension, batch-2*full_delay-max_full_shift))
    else:
        label = nd.empty((number_of_bathes, data_dimension, batch-2*full_delay))
    for i in range(number_of_bathes):
        data[i,:,:] = nd.array(X[:,i*batch:(i+1)*batch])
        for j in range(channels):
            if manual_shift:
                label[i,j*4:(j+1)*4,:] = nd.array(y[j*4:(j+1)*4,i*batch+full_delay+full_output_shifts[0]:(i+1)*batch-full_delay-full_output_shifts[1]])
            else:
                label[i,j*4:(j+1)*4,:] = nd.array(y[j*4:(j+1)*4,i*batch+full_delay:(i+1)*batch-full_delay])

    dataset = gluon.data.dataset.ArrayDataset(data, label)
    return gluon.data.DataLoader(dataset, batch_size=1, shuffle=enableShuffle), batch, number_of_bathes

# 5.3 Linear and Nonlinear layers

In [ ]:
class ComplexConvRandom(gluon.Block):
    def __init__(self, kernel_size, dimension, shift=np.array([[0,0]]), pol=2, enable_shift=True, sym=False, channels=1, strides=1, **kwargs):
        super(ComplexConvRandom, self).__init__(**kwargs)
        with self.name_scope():
            if len(shift) == 1 and np.sum(shift[0,:]) == 0:
                    shift = np.zeros((int(dimension/(2*pol)), 2),dtype=int)
            self.dimension = dimension
            self.shift = shift
            self.pol = pol
            self.kernel_size = kernel_size
            self.enable_shift = enable_shift
            self.sym = sym
            self.weight_re = self.params.get('weight_re', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size))
            self.weight_im = self.params.get('weight_im', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size))

    def forward(self, z):
        with z.context:
            chan = int(self.dimension/(2*self.pol))
            
            if self.sym:
                filt_re = nd.concat(self.weight_re.data(), nd.flip(self.weight_re.data(), axis=2)[:,:,1:], dim=2)
                filt_im = nd.concat(self.weight_im.data(), nd.flip(self.weight_im.data(), axis=2)[:,:,1:], dim=2)
                kernel = 2 * self.kernel_size - 1
            else:
                filt_re = self.weight_re.data()
                filt_im = self.weight_im.data()
                kernel = self.kernel_size
            
            y = nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) -\
            nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan)
           
            y = nd.concat(y, nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
            
            for i in range(2, 2*self.pol, 2):
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) - 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)

            v = nd.slice(y, begin=(None, 0, None), end=(None, self.dimension, None), step=(1, chan, 1))
            if self.enable_shift:
                v = v[:,:,self.shift[0,0]:v.shape[2]-self.shift[0,1]]
            for i in range(1, chan):
                u = nd.slice(y, begin=(None, i, None), end=(None, self.dimension, None), step=(1, chan, 1))
                if self.enable_shift:
                    u = u[:,:,self.shift[i,0]:u.shape[2]-self.shift[i,1]]
                v = nd.concat(v, u, dim=1)
            return v
        
class ComplexConvPredefined(gluon.Block):
    def __init__(self, weightsRe, weightsIm, kernel_size, dimension, shift=np.array([[0,0]]), pol=2, enable_shift=True, sym=False, channels=1, strides=1, **kwargs):
        super(ComplexConvPredefined, self).__init__(**kwargs)
        with self.name_scope():
            if len(shift) == 1 and np.sum(shift[0,:]) == 0:
                    shift = np.zeros((int(dimension/(2*pol)), 2),dtype=int)
            self.dimension = dimension
            self.shift = shift
            self.pol = pol
            self.kernel_size = kernel_size
            self.enable_shift = enable_shift
            self.sym = sym
            self.weight_re = self.params.get('weight_re', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size), init=CustomInit(weightsRe))
            self.weight_im = self.params.get('weight_im', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size), init=CustomInit(weightsIm))

    def forward(self, z):
        with z.context:
            chan = int(self.dimension/(2*self.pol))
            
            if self.sym:
                filt_re = nd.concat(self.weight_re.data(), nd.flip(self.weight_re.data(), axis=2)[:,:,1:], dim=2)
                filt_im = nd.concat(self.weight_im.data(), nd.flip(self.weight_im.data(), axis=2)[:,:,1:], dim=2)
                kernel = 2 * self.kernel_size - 1
            else:
                filt_re = self.weight_re.data()
                filt_im = self.weight_im.data()
                kernel = self.kernel_size
            
            y = nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) -\
            nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan)
           
            y = nd.concat(y, nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
            
            for i in range(2, 2*self.pol, 2):
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) - 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)

            v = nd.slice(y, begin=(None, 0, None), end=(None, self.dimension, None), step=(1, chan, 1))
            if self.enable_shift:
                v = v[:,:,self.shift[0,0]:v.shape[2]-self.shift[0,1]]
            for i in range(1, chan):
                u = nd.slice(y, begin=(None, i, None), end=(None, self.dimension, None), step=(1, chan, 1))
                if self.enable_shift:
                    u = u[:,:,self.shift[i,0]:u.shape[2]-self.shift[i,1]]
                v = nd.concat(v, u, dim=1)
            
            return v
        
class KerrActivationEnhanced_1ch(gluon.Block):
    def __init__(self, dimension, tensor_intra, nl_shifts, nl_coef=0.1, strides=1, **kwargs):
        super(KerrActivationEnhanced_1ch, self).__init__()
        chan = 1
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,memory_size), init=CustomInit(tensor_intra))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
                
        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*memory_size-1, num_group=self.chan)
        else:
            power = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=memory_size, num_group=self.chan)
             
        u = nd.cos(self.gamma.data() * power) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,2,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power) * z[:,3,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,3,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power) * z[:,2,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
class KerrActivationEnhanced_2ch(gluon.Block):
    def __init__(self, dimension, tensor_intra, tensor_inter, nl_shifts, nl_coef=0.1, strides=1, **kwargs):
        super(KerrActivationEnhanced_2ch, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,memory_size), init=CustomInit(tensor_intra))
            self.conv_inter = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=memory_size_nc1, strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        power_inter = power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]]
                
        power = nd.square(z[:,4:5,:]) +  nd.square(z[:,5:6,:]) + nd.square(z[:,6:7,:]) +  nd.square(z[:,7:8,:])
        power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[1,1,0]:power.shape[2]-self.nl_shifts[1,1,1]], dim=1)
        power_inter = nd.concat(power_inter, power[:,:,self.nl_shifts[1,0,0]:power.shape[2]-self.nl_shifts[1,0,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*memory_size-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=memory_size, num_group=self.chan)
        power_inter = self.conv_inter(power_inter)
        power = power_intra + nd.concat(power_inter[:,1:2,:], power_inter[:,0:1,:])
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, self.chan * 2):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
class KerrActivationEnhanced_4ch(gluon.Block):
    def __init__(self, dimension, tensor_intra, tensor_inter1, tensor_inter2, tensor_inter3, nl_shifts, nl_coef=0.1, strides=1, **kwargs):
        super(KerrActivationEnhanced_4ch, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,memory_size), init=CustomInit(tensor_intra))
            self.conv_inter1 = gluon.nn.Conv1D(channels=6, in_channels=6, groups=6, kernel_size=memory_size_nc1, strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter1))
            self.conv_inter2 = gluon.nn.Conv1D(channels=4, in_channels=4, groups=4, kernel_size=memory_size_nc2, strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter2))
            self.conv_inter3 = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=memory_size_nc3, strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter3))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        power_inter1 = power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]]
        power_inter2 = power[:,:,self.nl_shifts[0,2,0]:power.shape[2]-self.nl_shifts[0,2,1]]
        power_inter3 = power[:,:,self.nl_shifts[0,3,0]:power.shape[2]-self.nl_shifts[0,3,1]]
                
        for i in range(1, self.chan):
            power = nd.square(z[:,4*i:4*i+1,:]) +  nd.square(z[:,4*i+1:4*i+2,:]) + nd.square(z[:,4*i+2:4*i+3,:]) +  nd.square(z[:,4*i+3:4*i+4,:])
            for j in range(self.chan):
                if i == j:
                    power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[i,i,0]:power.shape[2]-self.nl_shifts[i,i,1]], dim=1)
                elif abs(i-j) == 1:
                    power_inter1 = nd.concat(power_inter1, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)
                elif abs(i-j) == 2:
                    power_inter2 = nd.concat(power_inter2, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)
                else:
                    power_inter3 = nd.concat(power_inter3, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*memory_size-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=memory_size, num_group=self.chan)
        power_inter1 = self.conv_inter1(power_inter1)
        power_inter2 = self.conv_inter2(power_inter2)
        power_inter3 = self.conv_inter3(power_inter3)
        power_inter = [power_inter1, power_inter2, power_inter3]
        
        power = power_intra
        power = power + nd.concat(power_inter1[:,1:2,:], power_inter1[:,0:1,:], power_inter2[:,0:1,:], power_inter3[:,0:1,:])
        power = power + nd.concat(power_inter2[:,2:3,:], power_inter1[:,3:4,:], power_inter1[:,2:3,:], power_inter2[:,1:2,:])
        power = power + nd.concat(power_inter3[:,1:2,:], power_inter2[:,3:4,:], power_inter1[:,5:6,:], power_inter1[:,4:5,:])
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, int(self.chan * 2)):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
class ComplexConvSeq(nn.Sequential):
    def __init__(self, chan, **kwargs):
        super(ComplexConvSeq, self).__init__(**kwargs)
        self.chan = chan
    def forward(self, z):
        v = self._children['0'](z)
        u = v
        for i in range(1, num_step):
            u = self._children[str(i)](u)
            if manual_shift:
                v = nd.concat(v[:,:,step_output_shifts[0]+filt_CDC_delay:v.shape[2]-filt_CDC_delay-step_output_shifts[1]], u, dim=0)
            else:
                v = nd.concat(v[:,:,filt_CDC_delay:v.shape[2]-filt_CDC_delay], u, dim=0)
        if last_step_length > 0:
            u = self._children[str(num_step)](u)
            if manual_shift:
                v = nd.concat(v[:,:,last_output_shifts[0]+filt_CDC_last_delay:v.shape[2]-filt_CDC_last_delay-last_output_shifts[1]], u, dim=0)
            else:
                v = nd.concat(v[:,:,filt_CDC_last_delay:v.shape[2]-filt_CDC_last_delay], u, dim=0)
        if FD_step_length > 0:
            u = self._children[str(len(self._children.items())-1)](u)
            v = nd.concat(v[:,:,filt_FD_delay:v.shape[2]-filt_FD_delay], u, dim=0)
        return v

# 5.4 CDC and FD filters

In [ ]:
def get_CDCfilt_step(data_real, width, length, shift, output_shift):
    if sym_filt:
        delay = width - 1
    else:
        delay = int((width-1)/2)
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    filt_Re = nd.empty((channels, 1, width), dtype=real_t)
    filt_Im = nd.empty((channels, 1, width), dtype=real_t)
    
    for i in range(channels):
        data_CDC = cd_operator(data, size, length, Fn[channel_numbers[i]], 'b')
        if manual_shift:
            data_CDC = data_CDC[delay+output_shift[0]:size-delay-output_shift[1]]
        else:
            data_CDC = data_CDC[delay:size-delay]
            
        net = gluon.nn.Sequential()
        with net.name_scope():
            net.add(ComplexConvRandom(kernel_size=width, dimension=2, shift=shift[i:i+1,:], pol=1, enable_shift=manual_shift, sym=sym_filt))

        net.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        if manual_shift:
            labels = nd.empty((1,2,size-2*delay-abs(np.sum(output_shift))))
        else:
            labels = nd.empty((1,2,size-2*delay))
        labels[:,0,:] = nd.array(np.real(data_CDC))
        labels[:,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 100
        while lossCond and epoch < 10000 and prev_loss > 1e-4:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = net(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                print('CDC filter step -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                lossCond = abs(train_loss.asscalar() - prev_loss) / prev_loss > 1e-6
                prev_loss = train_loss.asscalar()

        params = net.collect_params()
        filt_Re[i,:,:] = params[list(params)[0]].data().reshape(width,).asnumpy()
        filt_Im[i,:,:] = params[list(params)[1]].data().reshape(width,).asnumpy()
        
    return [filt_Re, filt_Im]


def get_FDfilt(data_real, width, length):
    delay = int((width-1)/2)
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    filtFD_Re = nd.empty((channels, 1, width), dtype=real_t)
    filtFD_Im = nd.empty((channels, 1, width), dtype=real_t)
    
    for i in range(channels):
        data_CDC = cd_operator(data, size, length, Fn[channel_numbers[i]], 'b')
        data_CDC = data_CDC[delay:size-delay]
    
        net = gluon.nn.Sequential()
        with net.name_scope():
            net.add(ComplexConvRandom(kernel_size=width, dimension=2, pol=1, enable_shift=False, sym=False))
    
        net.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()
    
        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        labels = nd.empty((1,2,size-2*delay))
        labels[:,0,:] = nd.array(np.real(data_CDC))
        labels[:,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)

        lossCond = True
        epoch = 0
        prev_loss = 100
        while lossCond and epoch < 10000 and prev_loss > 1e-5:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = net(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                print('FD filter -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                lossCond = abs(train_loss.asscalar() - prev_loss) / prev_loss > 1e-6
                prev_loss = train_loss.asscalar()

        params = net.collect_params()
        filtFD_Re[i,:,:] = params[list(params)[0]].data().reshape(width,).asnumpy()
        filtFD_Im[i,:,:] = params[list(params)[1]].data().reshape(width,).asnumpy()
        
    return [filtFD_Re, filtFD_Im]

def get_CDCfilt_JO(data_real, filt):
    
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    
    delay = num_step * filt_CDC_delay
    if last_step_length > 0:
        delay += filt_CDC_last_delay
    if FD_step_length > 0:
        delay += filt_FD_delay
    
    filtCDC_Re = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    filtCDC_Im = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    if last_step_length > 0:
        filtCDC_last_Re = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
        filtCDC_last_Im = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
    if FD_step_length > 0:
        filtFD_Re = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
        filtFD_Im = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
    
    for i in range(channels):
        
        netJO = ComplexConvSeq(i)
        for j in range(num_step):
            netJO.add(ComplexConvPredefined(weightsRe=filt[0][i,:,:], weightsIm=filt[1][i,:,:], kernel_size=filt_CDC_width, shift=step_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if last_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[2][i,:,:], weightsIm=filt[3][i,:,:], kernel_size=filt_CDC_last_width, shift=last_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if FD_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[len(filt)-2][i,:,:], weightsIm=filt[len(filt)-1][i,:,:], kernel_size=filt_FD_width, dimension=2, pol=1, enable_shift=False, sym=False))
        
        netJO.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(netJO.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        
        label_dimension = num_step
        if last_step_length > 0:
            label_dimension += 1
        if FD_step_length > 0:
            label_dimension += 1
        if manual_shift:
            labels = nd.empty((label_dimension,2,size-2*delay-max_full_shift))
        else:
            labels = nd.empty((label_dimension,2,size-2*delay))
            
        for j in range(num_step):
            data_CDC = cd_operator(data, size, (j + 1) * step_length, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[j,0,:] = nd.array(np.real(data_CDC))
            labels[j,1,:] = nd.array(np.imag(data_CDC))
        if last_step_length > 0:
            data_CDC = cd_operator(data, size, (j + 1) * step_length + last_step_length, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[num_step,0,:] = nd.array(np.real(data_CDC))
            labels[num_step,1,:] = nd.array(np.imag(data_CDC))
        if FD_step_length > 0:
            data_CDC = cd_operator(data, size, fiberL, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[label_dimension-1,0,:] = nd.array(np.real(data_CDC))
            labels[label_dimension-1,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 10
        while lossCond and epoch < 2000:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = netJO(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                print('CDC filter sequence -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                lossCond = train_loss.asscalar() > 1e-4
                if prev_loss < train_loss.asscalar():
                    break;
                else:
                    prev_loss = train_loss.asscalar()
                    
        params = netJO.collect_params()
        for j in range(num_step):
            filtCDC_Re[i,j,:] = params[list(params)[2*j]].data().reshape(filt_CDC_width,).asnumpy()
            filtCDC_Im[i,j,:] = params[list(params)[2*j+1]].data().reshape(filt_CDC_width,).asnumpy()
        if last_step_length > 0:
            filtCDC_last_Re[i,:,:] = params[list(params)[2*num_step]].data().reshape(filt_CDC_last_width,).asnumpy()
            filtCDC_last_Im[i,:,:] = params[list(params)[2*num_step+1]].data().reshape(filt_CDC_last_width,).asnumpy()
        if FD_step_length > 0:
            filtFD_Re[i,:,:] = params[list(params)[len(list(params))-1]].data().reshape(filt_FD_width,).asnumpy()
            filtFD_Im[i,:,:] = params[list(params)[len(list(params))-2]].data().reshape(filt_FD_width,).asnumpy()
         
    filters = [filtCDC_Re, filtCDC_Im]
    if last_step_length > 0:
            filters = filters + [filtCDC_last_Re, filtCDC_last_Im]
    if FD_step_length > 0:
            filters = filters + [filtFD_Re, filtFD_Im]
    return filters

# 6 Data preparation

In [ ]:
dir_path = create_metadata()

RX = np.empty((channels,data_size), dtype=complex_t)
RY = np.empty((channels,data_size), dtype=complex_t)
for i in range(channels):
    RX[i,:] = cd_operator(RX_CDC[channel_numbers[i],:], data_size, fiberL, Fn[channel_numbers[i]], 'f')
    RY[i,:] = cd_operator(RY_CDC[channel_numbers[i],:], data_size, fiberL, Fn[channel_numbers[i]], 'f')
    RX[i,:] /= np.sqrt(np.mean(abs(RX[i,:]) ** 2))
    RY[i,:] /= np.sqrt(np.mean(abs(RY[i,:]) ** 2))
    TX[channel_numbers[i],:] /= np.sqrt(np.mean(abs(TX[channel_numbers[i],:]) ** 2))
    TY[channel_numbers[i],:] /= np.sqrt(np.mean(abs(TY[channel_numbers[i],:]) ** 2))

del RX_CDC, RY_CDC

X = np.empty((data_dimension, data_size), dtype=real_t)
y = np.empty((data_dimension, data_size), dtype=real_t)

X[::4] = np.real(RX)
X[1::4] = np.imag(RX)
X[2::4] = np.real(RY)
X[3::4] = np.imag(RY)

y[::4] = np.real(TX[channel_numbers,:])
y[1::4] = np.imag(TX[channel_numbers,:])
y[2::4] = np.real(TY[channel_numbers,:])
y[3::4] = np.imag(TY[channel_numbers,:])

X_train = X[:,2000:train_data_size+2000]
y_train = y[:,2000:train_data_size+2000]

del TX, TY, RX, RY

# 7 CDC and FD filters coefficients calculation

In [ ]:
filt_CDC = get_CDCfilt_step(X_train[:2,:CDC_data_size], filt_CDC_width, step_length, step_shifts, step_output_shifts)
if last_step_length > 0:
    filt_CDC_last = get_CDCfilt_step(X_train[:2,:CDC_data_size], filt_CDC_last_width, last_step_length, last_shifts, last_output_shifts)
    filt_CDC = filt_CDC + filt_CDC_last
if FD_step_length > 0:
    filt_FD = get_FDfilt(X_train[:2,:CDC_data_size], filt_FD_width, FD_step_length)
    filt_CDC = filt_CDC + filt_FD

filt_CDC_JO = get_CDCfilt_JO(X_train[:2,:CDC_data_size], filt_CDC)

# 8 Create and compile model

In [ ]:
tensor_intra = nd.zeros((channels,1,memory_size))
tensor_intra[:,:,nl_mem] = nd.ones((channels,1))

if channels > 1:
    tensor_inter1 = 0.15 * nd.ones(((2*(channels-2)+2),1,memory_size_nc1))
if channels == 4:
    tensor_inter2 = 0.08 * nd.ones((4,1,memory_size_nc2))
    tensor_inter3 = 0.04 * nd.ones((2,1,memory_size_nc3))


net = gluon.nn.Sequential()
with net.name_scope():
    for i in range(num_step):
        net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[0][:,i:i+1,:], weightsIm=filt_CDC_JO[1][:,i:i+1,:], kernel_size=filt_CDC_width, shift=step_shifts, dimension=data_dimension, enable_shift=manual_shift, sym=sym_filt))
        if channels == 1:
            net.add(KerrActivationEnhanced_1ch(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef))
        if channels == 2:
            net.add(KerrActivationEnhanced_2ch(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef))
        if channels == 4:
            net.add(KerrActivationEnhanced_4ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_inter3=tensor_inter3, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef))
    if last_step_length > 0:
        net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[2], weightsIm=filt_CDC_JO[3], kernel_size=filt_CDC_last_width, shift=last_shifts, dimension=data_dimension, enable_shift=manual_shift, sym=sym_filt))
    if FD_step_length > 0:
        net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[len(filt_CDC_JO)-2], weightsIm=filt_CDC_JO[len(filt_CDC_JO)-1], kernel_size=filt_FD_width, dimension=data_dimension, enable_shift=False))

net.initialize(init=mx.init.Normal(sigma=0.05), ctx=ctx)

trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate})
mse = gluon.loss.L2Loss()

[train_dataloader, batch_train, number_of_bathes_train] = dataloader(X_train, y_train, batch_size, True)
del X_train, y_train

[all_data, batch_all, number_of_bathes_all] = dataloader(X, y, batch_size, False)
del X, y

# 2.2 Fit the model

In [ ]:
filename = os.path.join(dir_path, model_filename)
model_saved = False
min_loss = 1

decay_steps = 100
steps_wo_min = 0

for epoch in range(epochs):
    train_loss = nd.zeros(1, ctx=ctx)
    tic = time.time()
    for data, label in train_dataloader:
        data = data.as_in_context(ctx)
        label = label.as_in_context(ctx)
        with autograd.record():
            output = net(data)
            loss = mse(output, label)
            loss = nd.mean(loss)
        
        loss.backward()
        trainer.step(1)
        train_loss += loss.mean().asscalar()
        loss.wait_to_read()
    
    if (epoch + 1) % 10 == 0:
        print('epoch', epoch + 1, '-- loss', train_loss.asscalar()/number_of_bathes_train, '-- time', time.time()-tic)
    if min_loss > (train_loss.asscalar()/number_of_bathes_train):
        steps_wo_min = 0
        min_loss = train_loss.asscalar()/number_of_bathes_train
        if save_model_best:
            net.save_parameters(filename)
            model_saved = True
            print('epoch', epoch + 1, '-- Model saved', '-- loss', train_loss.asscalar()/number_of_bathes_train)
    else:
        steps_wo_min += 1
    
    if steps_wo_min >= decay_steps:
        steps_wo_min = 0
        lrate /= 2
        if lrate < 1e-5:
            break
        trainer.set_learning_rate(lrate)
        print('epoch', epoch + 1, '-- Learning rate', lrate)
    
output.wait_to_read()

# 3 BER calculation

In [ ]:
batch_trunc = batch_all - 2 * full_delay
if manual_shift:
    batch_trunc -= max_full_shift
y_pred = nd.empty([1, data_dimension, number_of_bathes_all * batch_trunc], dtype=real_t)
y = nd.empty([1, data_dimension, number_of_bathes_all * batch_trunc], dtype=real_t)

if save_model_best and model_saved:
    netTest = gluon.nn.Sequential()
    with netTest.name_scope():
        for i in range(num_step):
            netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[0][:,i:i+1,:], weightsIm=filt_CDC_JO[1][:,i:i+1,:], kernel_size=filt_CDC_width, shift=step_shifts, dimension=data_dimension, enable_shift=manual_shift, sym=sym_filt))
            if channels == 1:
                netTest.add(KerrActivationEnhanced_1ch(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
            if channels == 2:
                netTest.add(KerrActivationEnhanced_2ch(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
            if channels == 4:
                netTest.add(KerrActivationEnhanced_4ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_inter3=tensor_inter3, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
        if last_step_length > 0:
            netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[2], weightsIm=filt_CDC_JO[3], kernel_size=filt_CDC_last_width, shift=last_shifts, dimension=data_dimension, enable_shift=manual_shift, sym=sym_filt))
        if FD_step_length > 0:
            netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[len(filt_CDC_JO)-2], weightsIm=filt_CDC_JO[len(filt_CDC_JO)-1], kernel_size=filt_FD_width, dimension=data_dimension, enable_shift=False))

    netTest.load_parameters(filename, ctx=ctx)

for batch_idx, data in enumerate(all_data):
    data[0] = data[0].as_in_context(ctx)
    if save_model_best and model_saved:
        output = netTest(data[0])
    else:
        output = net(data[0])
    y_pred[:,:,batch_idx*batch_trunc:(batch_idx+1)*batch_trunc] = output.as_in_context(mx.cpu(0))
    y[:,:,batch_idx*batch_trunc:(batch_idx+1)*batch_trunc] = data[1]

output.wait_to_read() 
y_pred = y_pred.asnumpy()
y = y.asnumpy()

demap_data_size = 2 ** 20
mean_BER_NN = 0
for i in range(int(data_dimension/2)):
    a = demapper(y[:,2*i:2*(i+1),10000:10000+demap_data_size], y_pred[:,2*i:2*(i+1),10000:10000+demap_data_size])
    mean_BER_NN += calculate_ber(y_pred[:,2*i:2*(i+1),:] * a, y[:,2*i:2*(i+1),:])

mean_BER_CDC = (np.mean(BER_CDC_X[channel_numbers]) + np.mean(BER_CDC_Y[channel_numbers]))/2
mean_BER_DBP = (np.mean(BER_DBP_X[channel_numbers]) + np.mean(BER_DBP_Y[channel_numbers]))/2
mean_BER_NN /= data_dimension/2
print('BER CDC: ', mean_BER_CDC)
print('BER DBP Huawei: ', mean_BER_DBP)
print('Target BER: ', mean_BER_DBP*0.7)
print('BER NN: ', mean_BER_NN)

f = open(dir_path + 'metadata.md', 'a')
f.write('Epochs to decay: {}\n'.format(decay_steps))
f.write('\nEpochs total: {}\n'.format(epoch + 1))
f.write('Minimum loss: {}\n'.format(min_loss))

f.write('\nBER CDC: {}\n'.format(mean_BER_CDC))
f.write('BER DBP Huawei: {}\n'.format(mean_BER_DBP))
f.write('Target BER: {}\n'.format(mean_BER_DBP*0.7))
f.write('BER NN: {}\n'.format(mean_BER_NN))

if channels == 1:
    complexity = int(num_step * (0.5 * memory_size + 7 + 4 * filt_CDC_width))
           
if channels == 2:
    complexity = int(num_step * (0.5 * (memory_size + memory_size_nc1) + 7 + 4 * filt_CDC_width))
            
if channels == 4:
    complexity = int(num_step * (0.5 * (memory_size + 1.5 * memory_size_nc1 + memory_size_nc2 + 0.5 * memory_size_nc3) + 7 + 4 * filt_CDC_width))
        
if last_step_length > 0:
    complexity += 4 * filt_CDC_last_width
if FD_step_length > 0:
    complexity += 4 * filt_FD_width

            
f.write('\nComplexity: {}\n'.format(complexity))
print('Complexity: ', complexity)

f.close()